# **Checkpoint 1 - Fengyuan Guo - Sleep/Wake Activity Pattern Analysis**

## 1. Notebook Overview

The analysis focuses on understanding individual activity patterns, identifying potential sleep/wake periods, and examining how activity levels vary based on these patterns. This exploration will inform our group's approach to analyzing the relationship between circadian rhythms and physical activity.

**Driving Research Question:** Do they show variation in activity levels depending on their sleep/wake times or late-night activity?

**Selected Users:** 1503960366, 1624580081, 1644430081

**Data sources:** FitBit Fitness Tracker Data (Kaggle, CC0 Public Domain)

## 2. Initial Assumptions and Predictions

- Assumptions 1: I expect to see clear periods of inactivity typically during night time (12 AM - 6 AM).
- Assumption 2: Different users will show different patterns
- Assumption 3: Users with different sleep/wake times may show different activity levels




## 3. Data Importing and Preprocessing

- The first step is to load the three FitBit datasets and any librarymight beused. Then I want to understand their structure. This will help me verify that our selected users have complete and clear data for deep analysis..

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load 3 datasets
daily_steps = pd.read_csv('dailySteps_merged.csv')
hourly_steps = pd.read_csv('hourlySteps_merged.csv')
minute_steps = pd.read_csv('minuteStepsWide_merged.csv')

# Overview about data structure
print(daily_steps.head())
print(hourly_steps.head())
print(minute_steps.iloc[:5, :5])

## 4. Specific users data selection
- After load alldatasets and understanding their structure. I want to do data preprocessing. Since we only need 3 specific users for current EDA, I will filter data to only contain them.

In [ ]:
# Selected users
SELECTED_USERS = [
    1503960366,  # User 1
    1624580081,  # User 2
    1644430081   # User 3
]

print(f"Selected users for analysis: {SELECTED_USERS}")

# Verify these users exist
for user in SELECTED_USERS:
    daily_count = daily_steps[daily_steps['Id'] == user].shape[0]
    hourly_count = hourly_steps[hourly_steps['Id'] == user].shape[0]
    minute_count = minute_steps[minute_steps['Id'] == user].shape[0]
    print(f"User {user}: {daily_count} daily records, {hourly_count} hourly records, {minute_count} minute records")

# Filter datasets for selected users
daily_selected = daily_steps[daily_steps['Id'].isin(SELECTED_USERS)] if 'Id' in daily_steps.columns else pd.DataFrame()
hourly_selected = hourly_steps[hourly_steps['Id'].isin(SELECTED_USERS)] if 'Id' in hourly_steps.columns else pd.DataFrame()
minute_selected = minute_steps[minute_steps['Id'].isin(SELECTED_USERS)] if 'Id' in minute_steps.columns else pd.DataFrame()

### Analysis Function
- To improve efficiency and ensure consistent analysis for all 3 users, I will create 2 reusable functions to do exploration for both daily and minute step count with these 3 users.

### Daily step analysis
- The daily analysis function examines step count patterns over multiple days for each user. Including calculating basic statistics and identifying the date range of available data.

In [ ]:
def analyze_daily_steps(user_id, daily_data):

    user_daily = daily_data[daily_data['Id'] == user_id].copy()

    # Convert ActivityDay to datetime
    user_daily['ActivityDay'] = pd.to_datetime(user_daily['ActivityDay'])

    steps = user_daily['StepTotal']
    results = {
        'user_id': user_id,
        'days_of_data': len(user_daily),
        'avg_steps_per_day': steps.mean(),
        'max_steps': steps.max(),
        'min_steps': steps.min(),
        'std_steps': steps.std()
    }

    return results

### Minute step analysis
- The minute-level analysis is more complex. Each row contains 60 columns representing steps taken in each minute of an hour. So I will get each column then get results.

In [ ]:
def analyze_minute_steps(user_id, minute_data):

    user_minute = minute_data[minute_data['Id'] == user_id].copy()
    # Get step columns
    step_columns = [col for col in user_minute.columns if col.startswith('Steps')]


    # Flatten all step data (Convert to single list)
    all_steps = user_minute[step_columns].values.flatten()

    # Missing/NaN values
    valid_steps = all_steps[~pd.isna(all_steps)]
    missing_count = len(all_steps) - len(valid_steps)

    results = {
        'user_id': user_id,
        'total_minutes': len(all_steps),
        'non_zero_minutes': int(np.sum(valid_steps > 0)),
        'missing_data_points': missing_count,
        'avg_steps_per_minute': np.mean(valid_steps) if len(valid_steps) > 0 else 0,
        'max_steps_per_minute': np.max(valid_steps) if len(valid_steps) > 0 else 0,
        'min_steps_per_minute': np.min(valid_steps) if len(valid_steps) > 0 else 0,
        'steps_std': np.std(valid_steps) if len(valid_steps) > 0 else 0,
    }

    return results

### Result Function
- Since we have functions to analyze these data and get results now, I will create a result dispaly function.

In [ ]:
def print_user_analysis(user_id, daily_results, minute_results):
    print(f"ANALYSIS FOR USER {user_id}")
    print(f"{'-'*50}")

    if 'error' not in daily_results:
        print(f"\nDAILY STEP COUNT INFORMATION:")
        print(f"  Number of days of data: {daily_results.get('days_of_data', 'N/A')}")
        print(f"  Average steps per day: {daily_results.get('avg_steps_per_day', 0):.0f}")
        print(f"  Maximum step count: {daily_results.get('max_steps', 'N/A')}")
        print(f"  Minimum step count: {daily_results.get('min_steps', 'N/A')}")
        print(f"  Standard deviation of steps: {daily_results.get('std_steps', 0):.0f}")

    print(f"{'-'*50}")

    if 'error' not in minute_results:
        print(f"\nMINUTE STEP COUNT INFORMATION:")
        print(f"  Number of non-zero minutes: {minute_results.get('non_zero_minutes', 'N/A'):,}")
        print(f"  Missing data points: {minute_results.get('missing_data_points', 'N/A')}")
        print(f"  Average steps per minute: {minute_results.get('avg_steps_per_minute', 0):.2f}")
        print(f"  Maximum steps per minute: {minute_results.get('max_steps_per_minute', 'N/A')}")
        print(f"  Minimum steps per minute: {minute_results.get('min_steps_per_minute', 'N/A')}")
        print(f"  Standard deviation of steps per minute: {minute_results.get('steps_std', 0):.2f}")


## 6. Individual User Analysis

Now I will apply my analysis functions to each of the three selected users. This ensures consistent analysis that allows me observe individual differences in activity patterns.

### Analyzing User 1503960366
Here is the analysis result for first user

In [ ]:
# Analyze User 1 (1503960366)
user1_daily = analyze_daily_steps(SELECTED_USERS[0], daily_selected)
user1_minute = analyze_minute_steps(SELECTED_USERS[0], minute_selected)

print_user_analysis(SELECTED_USERS[0], user1_daily, user1_minute)

### Analyzing User 1624580081
Here is the analysis result for second user

In [ ]:
# Analyze User 2 (1624580081)
user2_daily = analyze_daily_steps(SELECTED_USERS[1], daily_selected)
user2_minute = analyze_minute_steps(SELECTED_USERS[1], minute_selected)

print_user_analysis(SELECTED_USERS[1], user2_daily, user2_minute)

### Analyzing User 1644430081
Here is the analysis result for third user

In [ ]:
# Analyze User 3 (1644430081)
user3_daily = analyze_daily_steps(SELECTED_USERS[2], daily_selected)
user3_minute = analyze_minute_steps(SELECTED_USERS[2], minute_selected)

print_user_analysis(SELECTED_USERS[2], user3_daily, user3_minute)

## 7. EDA about sleep/wake patterns about hourly data
Based on our research question, I need to understand how activity levels vary with sleep/wake times. To approach this, I will analyze hourly activity patterns which should reveal periods of low activity (likely sleep) and high activity periods.

### 7.1 Analyzing functions
The following function processes hourly data to identify potential sleep/wake cycles by examining activity levels throughout the day.

In [ ]:
# Advanced sleep/wake pattern analysis using hourly data
def explore_temporal_patterns(user_id, hourly_data):
    user_hourly = hourly_data[hourly_data['Id'] == user_id].copy()

    # Convert to datetime and extract time components
    user_hourly['DateTime'] = pd.to_datetime(user_hourly['ActivityHour'])
    user_hourly['Hour'] = user_hourly['DateTime'].dt.hour
    user_hourly['Date'] = user_hourly['DateTime'].dt.date

    # Calculate average steps by hour across all days
    hourly_avg = user_hourly.groupby('Hour')['StepTotal'].mean().reset_index()

    print(f"\nTEMPORAL PATTERN ANALYSIS FOR USER {user_id}:")
    print(f"  Total hourly records: {len(user_hourly)}")
    print(f"  Date range: {user_hourly['Date'].min()} to {user_hourly['Date'].max()}")

    # Identify likely sleep hours (hours with consistently low activity)
    low_activity_hours = hourly_avg[hourly_avg['StepTotal'] < 100]['Hour'].tolist()
    peak_activity_hour = hourly_avg.loc[hourly_avg['StepTotal'].idxmax(), 'Hour']

    print(f"  Peak activity hour: {peak_activity_hour}:00 (avg {hourly_avg['StepTotal'].max():.0f} steps)")
    print(f"  Low activity hours (likely sleep): {low_activity_hours}")

    # Simple sleep/wake pattern detection
    if low_activity_hours:
        night_hours = [h for h in low_activity_hours if h >= 22 or h <= 6]
        if night_hours:
            print(f"  Likely sleep pattern: Traditional (low activity 10PM-6AM)")
        else:
            print(f"  Likely sleep pattern: Non-traditional (low activity during day)")

    return user_hourly

### 7.2 Applying to Each User
Now I will apply this analysis function to understand each user's potential sleep/wake patterns. This is important for answering our research question about activity variation based on sleep/wake times.

In [ ]:
# Explore patterns for each user
for user_id in SELECTED_USERS:
    user_hourly_data = explore_temporal_patterns(user_id, hourly_selected)

    # Visualization
    if user_hourly_data is not None:
        plt.figure(figsize=(12, 4))
        hourly_avg = user_hourly_data.groupby('Hour')['StepTotal'].mean()
        plt.plot(hourly_avg.index, hourly_avg.values, marker='o', linewidth=2, markersize=6)
        plt.title(f'Average Hourly Step Count - User {user_id}')
        plt.xlabel('Hour of Day')
        plt.ylabel('Average Steps')
        plt.grid(True, alpha=0.3)
        plt.xticks(range(0, 24, 2))

        # Highlight potential sleep hours (low activity)
        sleep_threshold = hourly_avg.quantile(0.25)  # Bottom 25%
        plt.axhline(y=sleep_threshold, color='red', linestyle='--', alpha=0.7,
                   label=f'Sleep threshold (~{sleep_threshold:.0f} steps)')
        plt.legend()
        plt.tight_layout()
        plt.show()

## 8. Sumary of findings

### 8.1 Learned from Data Exploration
**Sleep/Wake Insights:**
- User 1503960366: Peak at 6PM - evening active type. User 1624580081: Peak at 10AM - morning active type. User 1644430081: Peak at 7PM - evening active type.
- Users 1503960366 and 1644430081 maintained higher evening activity levels compared to User 1624580081.
- User 1503960366: Sleep hours 1AM-7AM (7 hours). User 1624580081: Sleep hours 11PM-5AM (7 hours). User 1644430081: Sleep hours 11PM-7AM (9 hours).

**Activity Level Patterns:**
- User 1503960366 was the most active with an average of 12,117 steps per day, followed by User 1644430081 (7,283 steps/day), and User 1624580081 (5,744 steps/day).
- User 1624580081 showed the highest day-to-day variation (std = 6,177 steps). User 1503960366 showed moderate consistency (std = 3,052), while User 1644430081 demonstrated the most consistent daily patterns (std = 4,325).
- User 1503960366 was active 19.2% of recorded time, User 1644430081 was active 14.6% of time, while User 1624580081 was active only 8.4% of time.

**Data Quality Aspects:**
- All three users had excellent data coverage with 30-31 days of daily data, 708-736 hourly records, and 684-729 minute-level records.
- Each user showed consistent data collection across all three datasets (daily, hourly, minute).


### 8.2 How this related to our Driving Question
**Research Question:** Do they show variation in activity levels depending on their sleep/wake times or late-night activity?

**Key insights**
- Users with later peak activity times showed higher overall daily activity levels compared to the morning-peak user.
- User 1644430081, who showed the longest sleep period (9 hours), maintained moderate-to-high activity levels (7,283 steps/day), opposite to the assumption that longer sleep necessarily correlates with lower activity.
- The visualizations showed that users bot only differ in total activity, but also in how that activity is distributed throughout the day.


### 8.3 Initial Assumptions Results
- Clear periods of inactivity during nighttime hours were evident for all users.
- Individual users showed different patterns, with consistent differences in peak activity timing.
- The minute analysis revealed that active time percentages were much low, indicating that most time is not walking even for active individuals


### 8.4 Next Steps
1. Outlier Detection and Analysis: Develop approaches for identifying and analyzing extreme activity days to determine if they represent behavioral variation or data collection issues.
2. Develop Robust Classification: Create standardized metrics to classify users as different types based on their peak activity timing and sleep/wake patterns.
3. Activity-Sleep Correlations: Conduct statistical analysis to test whether users with different peak activity times show significantly different total daily activity levels, activity distribution patterns, or sleep duration indicators.


### 8.5 What I Learned

**Technical Learning:**
- Enhanced skills in pandas data processing and datetime handling
- Improved data visualization techniques
- Better understanding of exploratory data analysis workflows

**Others Learning:**
- Importance of systematic data exploration before hypothesis testing
- Importance about literature programming for work/research
- Creating reusable analysis functions for comparative studies

### 8.6 Summary
This individual checkpoint has provided a clear foundation for understanding our dataset and revealed clear evidence that users do show variation in activity levels depending on their sleep/wake times.